# 02 — Unir integrado preliminar canónico

**Proyecto:** calidad de datos bibliográficos de artículos UNAM en Ciencias de la Computación.  
**Objetivo:** unir los 8 archivos canónicos por fuente ya clasificados por `Area`, sin deduplicar, sin fusionar registros y sin limpieza profunda.

Esta libreta genera:

- `02_modelo canonico/03_union/integrado_preliminar_canonico.csv`
- `02_modelo canonico/03_union/integrado_preliminar_canonico_trazabilidad.csv`
- `02_modelo canonico/03_union/resumen_integrado_preliminar.md`
- `outputs/reportes/reporte_integracion_preliminar.xlsx`

In [1]:
from pathlib import Path
from datetime import datetime
import re
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

## 1. Configuración de rutas y modelo canónico

La libreta detecta la raíz del proyecto automáticamente. Está preparada para la estructura actual:

```text
C:\Users\hazar\Desktop\PROYECTO\02_modelo canonico\02_por_area
C:\Users\hazar\Desktop\PROYECTO\02_modelo canonico\03_union
```

También incluye rutas alternativas por si en otra copia del proyecto se usa `03_modelo_canonico`.

In [2]:
COLUMNAS_CANONICAS = [
    "indice",
    "Titulo",
    "Año",
    "Autor_norm",
    "Afiliacion1",
    "Afiliacion2",
    "ISBN",
    "ISSN",
    "Doi",
    "URL",
    "Area",
    "Subarea",
    "Keywords",
    "Abstract",
]

COLUMNAS_TRAZABILIDAD = [
    "_fuente",
    "_archivo_origen",
    "_area_lote",
    "_id_fila_origen",
]

AREAS_VALIDAS = {"ISBD", "CC", "IA", "TC", "SIAV", "RS"}

ARCHIVOS_ESPERADOS = [
    "acm_canonico_2024-03_2025-12_area_clasificada.csv",
    "ebsco_canonico_2024-03_2025-12_area_clasificada.csv",
    "engineering_village_canonico_2024-03_2025-12_area_clasificada.csv",
    "ieee_xplore_canonico_2024-03_2025-12_area_clasificada.csv",
    "proquest_canonico_2024-03_2025-12_area_clasificada.csv",
    "sciencedirect_canonico_2024-03_2025-12_area_clasificada.csv",
    "scopus_canonico_2024-03_2025-12_area_clasificada.csv",
    "web_of_science_canonico_2024-03_2025-12_area_clasificada.csv",
]


def encontrar_raiz_proyecto() -> Path:
    """Busca la raíz del proyecto desde el directorio actual hacia arriba."""
    cwd = Path.cwd().resolve()
    candidatos = [cwd] + list(cwd.parents)

    for ruta in candidatos:
        marcadores = [
            ruta / "notebooks",
            ruta / "02_modelo canonico",
            ruta / "03_modelo_canonico",
            ruta / "01_recoleccion_manual",
        ]
        if any(m.exists() for m in marcadores):
            return ruta

    ruta_windows = Path(r"C:\Users\hazar\Desktop\PROYECTO")
    if ruta_windows.exists():
        return ruta_windows

    return cwd.parent if cwd.name.lower() == "notebooks" else cwd


ROOT = encontrar_raiz_proyecto()

ENTRADA_CANDIDATOS = [
    ROOT / "02_modelo canonico" / "02_por_area",
    ROOT / "03_modelo_canonico" / "02_por_area",
    ROOT / "03_modelo_canonico",
]

ENTRADA_DIR = next((p for p in ENTRADA_CANDIDATOS if p.exists()), ENTRADA_CANDIDATOS[0])

# Carpeta principal solicitada por el proyecto actual.
if (ROOT / "02_modelo canonico").exists() or not (ROOT / "03_modelo_canonico").exists():
    UNION_DIR = ROOT / "02_modelo canonico" / "03_union"
else:
    UNION_DIR = ROOT / "03_modelo_canonico" / "03_union"

REPORTES_DIR = ROOT / "outputs" / "reportes"
UNION_DIR.mkdir(parents=True, exist_ok=True)
REPORTES_DIR.mkdir(parents=True, exist_ok=True)

SALIDA_INTEGRADO = UNION_DIR / "integrado_preliminar_canonico.csv"
SALIDA_TRAZABILIDAD = UNION_DIR / "integrado_preliminar_canonico_trazabilidad.csv"
SALIDA_RESUMEN_MD = UNION_DIR / "resumen_integrado_preliminar.md"
SALIDA_REPORTE_XLSX = REPORTES_DIR / "reporte_integracion_preliminar.xlsx"

print("Raíz del proyecto:", ROOT)
print("Entrada:", ENTRADA_DIR)
print("Salida unión:", UNION_DIR)
print("Reporte Excel:", SALIDA_REPORTE_XLSX)

Raíz del proyecto: C:\Users\hazar\Desktop\PROYECTO
Entrada: C:\Users\hazar\Desktop\PROYECTO\02_modelo canonico\02_por_area
Salida unión: C:\Users\hazar\Desktop\PROYECTO\02_modelo canonico\03_union
Reporte Excel: C:\Users\hazar\Desktop\PROYECTO\outputs\reportes\reporte_integracion_preliminar.xlsx


## 2. Funciones auxiliares

Estas funciones solo hacen normalizaciones estructurales mínimas:

- leer CSV de forma robusta;
- quitar filas completamente vacías;
- crear columnas canónicas faltantes si aparecen;
- separar columnas extra del integrado final;
- agregar trazabilidad;
- reportar problemas sin corregirlos.

In [3]:
def leer_csv_robusto(ruta: Path) -> tuple[pd.DataFrame, str, str]:
    """Lee un CSV probando combinaciones comunes de encoding y separador.

    No altera el contenido del archivo. Usa dtype=str para preservar metadatos bibliográficos.
    """
    intentos = [
        ("utf-8-sig", ","),
        ("utf-8-sig", None),
        ("utf-8", ","),
        ("cp1252", ","),
        ("latin1", ","),
        ("utf-8-sig", ";"),
        ("utf-8-sig", "\t"),
    ]
    errores = []

    for encoding, sep in intentos:
        try:
            kwargs = {
                "encoding": encoding,
                "dtype": str,
                "keep_default_na": False,
                "na_filter": False,
                "low_memory": False,
            }
            if sep is None:
                kwargs.update({"sep": None, "engine": "python"})
            else:
                kwargs.update({"sep": sep})

            df = pd.read_csv(ruta, **kwargs)
            if len(df.columns) >= 1:
                return df, encoding, ("detectado" if sep is None else sep)
        except Exception as exc:
            errores.append(f"encoding={encoding}, sep={sep}: {exc}")

    detalle = "\n".join(errores[-5:])
    raise ValueError(f"No se pudo leer el archivo {ruta.name}. Últimos errores:\n{detalle}")


def quitar_filas_completamente_vacias(df: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    """Elimina únicamente filas donde todas las celdas están vacías o con espacios."""
    if df.empty:
        return df.copy(), 0
    mascara_vacia = df.astype(str).apply(lambda col: col.str.strip().eq("")).all(axis=1)
    n_vacias = int(mascara_vacia.sum())
    return df.loc[~mascara_vacia].reset_index(drop=True), n_vacias


def inferir_fuente(nombre_archivo: str) -> str:
    """Infiere la fuente desde el nombre del archivo."""
    nombre = nombre_archivo.lower()
    fuentes = [
        "engineering_village",
        "web_of_science",
        "ieee_xplore",
        "sciencedirect",
        "proquest",
        "scopus",
        "ebsco",
        "acm",
    ]
    for fuente in fuentes:
        if nombre.startswith(fuente):
            return fuente
    return nombre.split("_canonico")[0]


def inferir_area_lote_desde_indice(valor_indice: str, area_respaldo: str = "") -> str:
    """Extrae el área original del lote desde índices como SCOPUS_CC_0001 o WOS_ISBD_0031.

    Si no se puede inferir, usa el valor de Area como respaldo.
    """
    texto = str(valor_indice).upper().strip()
    for area in ["ISBD", "SIAV", "SIAAV", "CC", "IA", "TC", "RS"]:
        if re.search(rf"(^|_){area}(_|$)", texto):
            return "SIAV" if area == "SIAAV" else area
    return str(area_respaldo).strip()


def serie_area_lote(df: pd.DataFrame) -> pd.Series:
    """Construye _area_lote fila por fila usando indice como fuente principal."""
    indices = df["indice"] if "indice" in df.columns else pd.Series([""] * len(df))
    areas = df["Area"] if "Area" in df.columns else pd.Series([""] * len(df))
    return pd.Series(
        [inferir_area_lote_desde_indice(idx, area) for idx, area in zip(indices, areas)],
        index=df.index,
    )


def df_a_markdown(df: pd.DataFrame) -> str:
    """Convierte una tabla pequeña a Markdown sin depender de tabulate."""
    if df is None or df.empty:
        return "_Sin registros._"

    tabla = df.copy().astype(str)
    columnas = list(tabla.columns)

    def limpiar(valor: str) -> str:
        return str(valor).replace("|", "/").replace("\n", " ")

    lineas = []
    lineas.append("| " + " | ".join(limpiar(c) for c in columnas) + " |")
    lineas.append("| " + " | ".join("---" for _ in columnas) + " |")
    for _, fila in tabla.iterrows():
        lineas.append("| " + " | ".join(limpiar(fila[c]) for c in columnas) + " |")
    return "\n".join(lineas)

## 3. Localizar los 8 archivos canónicos clasificados

Se priorizan exactamente los nombres corregidos con sufijo `_area_clasificada.csv`. Si no se encuentran esos nombres, la libreta hace un respaldo buscando cualquier archivo `*_area_clasificada.csv` en la carpeta de entrada.

In [4]:
archivos_esperados = [ENTRADA_DIR / nombre for nombre in ARCHIVOS_ESPERADOS]
archivos_encontrados = [ruta for ruta in archivos_esperados if ruta.exists()]
archivos_faltantes = [ruta.name for ruta in archivos_esperados if not ruta.exists()]

if not archivos_encontrados:
    archivos_encontrados = sorted(ENTRADA_DIR.glob("*_area_clasificada.csv"))

print(f"Archivos encontrados: {len(archivos_encontrados)}")
for ruta in archivos_encontrados:
    print("-", ruta.name)

if archivos_faltantes:
    print("\nArchivos esperados no encontrados:")
    for nombre in archivos_faltantes:
        print("-", nombre)

if not archivos_encontrados:
    raise FileNotFoundError(
        f"No se encontraron archivos *_area_clasificada.csv en {ENTRADA_DIR}. "
        "Revisa la ruta de entrada y los nombres de los archivos."
    )

Archivos encontrados: 8
- acm_canonico_2024-03_2025-12_area_clasificada.csv
- ebsco_canonico_2024-03_2025-12_area_clasificada.csv
- engineering_village_canonico_2024-03_2025-12_area_clasificada.csv
- ieee_xplore_canonico_2024-03_2025-12_area_clasificada.csv
- proquest_canonico_2024-03_2025-12_area_clasificada.csv
- sciencedirect_canonico_2024-03_2025-12_area_clasificada.csv
- scopus_canonico_2024-03_2025-12_area_clasificada.csv
- web_of_science_canonico_2024-03_2025-12_area_clasificada.csv


## 4. Leer, validar estructura y preparar cada archivo

Cada archivo se convierte temporalmente a dos versiones:

1. **versión canónica:** solo las 14 columnas del modelo;
2. **versión con trazabilidad:** las 14 columnas más `_fuente`, `_archivo_origen`, `_area_lote`, `_id_fila_origen`.

No se deduplican registros y no se fusionan filas.

In [5]:
def preparar_archivo_canonico(ruta: Path) -> dict:
    df_original, encoding_usado, sep_usado = leer_csv_robusto(ruta)

    columnas_originales = list(df_original.columns)
    df = df_original.copy()
    df.columns = [str(c).strip() for c in df.columns]
    columnas_despues_strip = list(df.columns)

    columnas_renombradas_por_espacios = [
        f"{antes} -> {despues}"
        for antes, despues in zip(columnas_originales, columnas_despues_strip)
        if str(antes) != str(despues)
    ]

    filas_leidas = len(df)
    df, filas_vacias_eliminadas = quitar_filas_completamente_vacias(df)

    columnas_faltantes = [c for c in COLUMNAS_CANONICAS if c not in df.columns]
    columnas_extra = [c for c in df.columns if c not in COLUMNAS_CANONICAS]

    for columna in columnas_faltantes:
        df[columna] = ""

    columnas_canonicas_encontradas = [c for c in df.columns if c in COLUMNAS_CANONICAS]
    orden_canonico_antes = columnas_canonicas_encontradas == COLUMNAS_CANONICAS

    df_canonico = df[COLUMNAS_CANONICAS].copy()

    fuente = inferir_fuente(ruta.name)
    ids_origen = df_canonico["indice"].astype(str).str.strip()
    ids_respaldo = pd.Series(
        [f"{ruta.stem}::fila_{i:05d}" for i in range(1, len(df_canonico) + 1)],
        index=df_canonico.index,
    )

    df_traza = df_canonico.copy()
    df_traza["_fuente"] = fuente
    df_traza["_archivo_origen"] = ruta.name
    df_traza["_area_lote"] = serie_area_lote(df_canonico)
    df_traza["_id_fila_origen"] = ids_origen.where(ids_origen.ne(""), ids_respaldo)

    areas_limpias = df_canonico["Area"].astype(str).str.strip()
    areas_invalidas = sorted([a for a in areas_limpias.unique() if a and a not in AREAS_VALIDAS])

    resumen_archivo = {
        "archivo": ruta.name,
        "fuente": fuente,
        "encoding": encoding_usado,
        "separador": sep_usado,
        "filas_leidas": filas_leidas,
        "filas_vacias_eliminadas": filas_vacias_eliminadas,
        "filas_integrables": len(df_canonico),
        "num_columnas_originales": len(columnas_originales),
        "num_columnas_canonicas": len(COLUMNAS_CANONICAS),
        "columnas_faltantes": "; ".join(columnas_faltantes),
        "columnas_extra": "; ".join(columnas_extra),
        "columnas_renombradas_por_espacios": "; ".join(columnas_renombradas_por_espacios),
        "orden_canonico_antes": orden_canonico_antes,
        "orden_canonico_despues": list(df_canonico.columns) == COLUMNAS_CANONICAS,
        "areas_invalidas": "; ".join(areas_invalidas),
        "titulos_vacios": int(df_canonico["Titulo"].astype(str).str.strip().eq("").sum()),
        "areas_vacias": int(areas_limpias.eq("").sum()),
        "doi_vacios": int(df_canonico["Doi"].astype(str).str.strip().eq("").sum()),
        "indice_vacios": int(df_canonico["indice"].astype(str).str.strip().eq("").sum()),
    }

    return {
        "df_canonico": df_canonico,
        "df_traza": df_traza,
        "resumen": resumen_archivo,
        "faltantes": columnas_faltantes,
        "extras": columnas_extra,
    }


resultados = [preparar_archivo_canonico(ruta) for ruta in archivos_encontrados]

resumen_archivos = pd.DataFrame([r["resumen"] for r in resultados])
resumen_archivos

,archivo,fuente,encoding,separador,filas_leidas,filas_vacias_eliminadas,filas_integrables,num_columnas_originales,num_columnas_canonicas,columnas_faltantes,columnas_extra,columnas_renombradas_por_espacios,orden_canonico_antes,orden_canonico_despues,areas_invalidas,titulos_vacios,areas_vacias,doi_vacios,indice_vacios
0,acm_canonico_2024-03_2025-12_area_clasificada.csv,acm,utf-8-sig,",",57,0,57,14,14,,,,True,True,,0,0,0,0
1,ebsco_canonico_2024-03_2025-12_area_clasificada.csv,ebsco,utf-8-sig,",",129,0,129,14,14,,,,True,True,,0,0,0,0
2,engineering_village_canonico_2024-03_2025-12_area_clasificada.csv,engineering_village,utf-8-sig,",",142,0,142,14,14,,,,True,True,,0,0,6,0
3,ieee_xplore_canonico_2024-03_2025-12_area_clasificada.csv,ieee_xplore,utf-8-sig,",",158,0,158,14,14,,,,True,True,,0,0,0,0
4,proquest_canonico_2024-03_2025-12_area_clasificada.csv,proquest,utf-8-sig,",",19,0,19,14,14,,,,True,True,,0,0,0,0
5,sciencedirect_canonico_2024-03_2025-12_area_clasificada.csv,sciencedirect,utf-8-sig,",",43,0,43,14,14,,,,True,True,,0,0,0,0
6,scopus_canonico_2024-03_2025-12_area_clasificada.csv,scopus,utf-8-sig,",",561,0,561,14,14,,,,True,True,,0,0,28,0
7,web_of_science_canonico_2024-03_2025-12_area_clasificada.csv,web_of_science,utf-8-sig,",",362,0,362,14,14,,,,True,True,,0,0,19,0


## 5. Concatenar fuentes

Esta etapa solo concatena. No deduplica, no fusiona y no limpia contenido.

In [6]:
frames_canonicos = [r["df_canonico"] for r in resultados]
frames_traza = [r["df_traza"] for r in resultados]

integrado_preliminar = pd.concat(frames_canonicos, ignore_index=True)
integrado_trazabilidad = pd.concat(frames_traza, ignore_index=True)

# Validaciones duras de estructura después de concatenar.
assert list(integrado_preliminar.columns) == COLUMNAS_CANONICAS, "El integrado sin trazabilidad no tiene el orden canónico."
assert list(integrado_trazabilidad.columns) == COLUMNAS_CANONICAS + COLUMNAS_TRAZABILIDAD, "El integrado con trazabilidad no tiene el orden esperado."

print("Total de filas integradas:", len(integrado_preliminar))
print("Columnas del integrado sin trazabilidad:", list(integrado_preliminar.columns))
print("Columnas del integrado con trazabilidad:", list(integrado_trazabilidad.columns))

Total de filas integradas: 1471
Columnas del integrado sin trazabilidad: ['indice', 'Titulo', 'Año', 'Autor_norm', 'Afiliacion1', 'Afiliacion2', 'ISBN', 'ISSN', 'Doi', 'URL', 'Area', 'Subarea', 'Keywords', 'Abstract']
Columnas del integrado con trazabilidad: ['indice', 'Titulo', 'Año', 'Autor_norm', 'Afiliacion1', 'Afiliacion2', 'ISBN', 'ISSN', 'Doi', 'URL', 'Area', 'Subarea', 'Keywords', 'Abstract', '_fuente', '_archivo_origen', '_area_lote', '_id_fila_origen']


## 6. Tablas de validación

Se generan tablas para revisar la integración sin corregir problemas de calidad bibliográfica.

In [7]:
conteo_fuente = (
    integrado_trazabilidad.groupby("_fuente", dropna=False)
    .size()
    .reset_index(name="filas")
    .sort_values("_fuente")
)

conteo_area = (
    integrado_preliminar.assign(Area=integrado_preliminar["Area"].astype(str).str.strip().replace("", "[vacía]"))
    .groupby("Area", dropna=False)
    .size()
    .reset_index(name="filas")
    .sort_values("Area")
)

conteo_fuente_area = (
    integrado_trazabilidad.assign(Area=integrado_trazabilidad["Area"].astype(str).str.strip().replace("", "[vacía]"))
    .groupby(["_fuente", "Area"], dropna=False)
    .size()
    .reset_index(name="filas")
    .sort_values(["_fuente", "Area"])
)

columnas_faltantes = []
for r in resultados:
    archivo = r["resumen"]["archivo"]
    fuente = r["resumen"]["fuente"]
    faltantes = r["faltantes"]
    if faltantes:
        for col in faltantes:
            columnas_faltantes.append({"archivo": archivo, "fuente": fuente, "columna_faltante_creada": col})
    else:
        columnas_faltantes.append({"archivo": archivo, "fuente": fuente, "columna_faltante_creada": ""})
columnas_faltantes_df = pd.DataFrame(columnas_faltantes)

columnas_extra = []
for r in resultados:
    archivo = r["resumen"]["archivo"]
    fuente = r["resumen"]["fuente"]
    extras = r["extras"]
    if extras:
        for col in extras:
            columnas_extra.append({"archivo": archivo, "fuente": fuente, "columna_extra_separada": col})
    else:
        columnas_extra.append({"archivo": archivo, "fuente": fuente, "columna_extra_separada": ""})
columnas_extra_df = pd.DataFrame(columnas_extra)

orden_canonico = resumen_archivos[[
    "archivo",
    "fuente",
    "orden_canonico_antes",
    "orden_canonico_despues",
    "num_columnas_originales",
    "num_columnas_canonicas",
]].copy()

total_esperado = int(resumen_archivos["filas_integrables"].sum())
total_integrado = int(len(integrado_preliminar))

totales = pd.DataFrame([
    {"metrica": "archivos_esperados", "valor": len(ARCHIVOS_ESPERADOS)},
    {"metrica": "archivos_encontrados", "valor": len(archivos_encontrados)},
    {"metrica": "archivos_faltantes", "valor": len(archivos_faltantes)},
    {"metrica": "filas_esperadas", "valor": total_esperado},
    {"metrica": "filas_integradas", "valor": total_integrado},
    {"metrica": "coinciden_filas_esperadas_integradas", "valor": total_esperado == total_integrado},
    {"metrica": "columnas_integrado_sin_trazabilidad", "valor": len(integrado_preliminar.columns)},
    {"metrica": "columnas_integrado_con_trazabilidad", "valor": len(integrado_trazabilidad.columns)},
])

problemas = []
for nombre in archivos_faltantes:
    problemas.append({"tipo": "archivo_faltante", "archivo": nombre, "detalle": "Archivo esperado no encontrado en carpeta de entrada."})

for _, fila in resumen_archivos.iterrows():
    archivo = fila["archivo"]
    if fila["filas_vacias_eliminadas"] > 0:
        problemas.append({"tipo": "filas_vacias_eliminadas", "archivo": archivo, "detalle": fila["filas_vacias_eliminadas"]})
    if str(fila["columnas_faltantes"]).strip():
        problemas.append({"tipo": "columnas_faltantes_creadas", "archivo": archivo, "detalle": fila["columnas_faltantes"]})
    if str(fila["columnas_extra"]).strip():
        problemas.append({"tipo": "columnas_extra_separadas", "archivo": archivo, "detalle": fila["columnas_extra"]})
    if not bool(fila["orden_canonico_antes"]):
        problemas.append({"tipo": "orden_canonico_no_original", "archivo": archivo, "detalle": "Se reordenó al modelo canónico."})
    if str(fila["areas_invalidas"]).strip():
        problemas.append({"tipo": "areas_invalidas", "archivo": archivo, "detalle": fila["areas_invalidas"]})
    if int(fila["titulos_vacios"]) > 0:
        problemas.append({"tipo": "titulos_vacios", "archivo": archivo, "detalle": fila["titulos_vacios"]})
    if int(fila["areas_vacias"]) > 0:
        problemas.append({"tipo": "areas_vacias", "archivo": archivo, "detalle": fila["areas_vacias"]})
    if int(fila["indice_vacios"]) > 0:
        problemas.append({"tipo": "indice_vacios", "archivo": archivo, "detalle": fila["indice_vacios"]})

# Señales de perfilado; no se corrigen aquí.
duplicados_indice = int(integrado_preliminar["indice"].astype(str).str.strip().duplicated().sum())
if duplicados_indice > 0:
    problemas.append({"tipo": "indices_duplicados", "archivo": "integrado", "detalle": duplicados_indice})

problemas_df = pd.DataFrame(problemas, columns=["tipo", "archivo", "detalle"])

print("Filas por fuente")
display(conteo_fuente)
print("\nFilas por área")
display(conteo_area)
print("\nTotales")
display(totales)
print("\nProblemas detectados sin corregir")
display(problemas_df)

Filas por fuente


,_fuente,filas
0,acm,57
1,ebsco,129
2,engineering_village,142
3,ieee_xplore,158
4,proquest,19
5,sciencedirect,43
6,scopus,561
7,web_of_science,362



Filas por área


,Area,filas
0,CC,141
1,IA,403
2,ISBD,110
3,RS,189
4,SIAV,419
5,TC,209



Totales


,metrica,valor
0,archivos_esperados,8
1,archivos_encontrados,8
2,archivos_faltantes,0
3,filas_esperadas,1471
4,filas_integradas,1471
5,coinciden_filas_esperadas_integradas,True
6,columnas_integrado_sin_trazabilidad,14
7,columnas_integrado_con_trazabilidad,18



Problemas detectados sin corregir


,tipo,archivo,detalle


## 7. Guardar CSV, reporte Excel y resumen Markdown

In [8]:
# CSV principal: integrado preliminar con solo las 14 columnas canónicas.
integrado_preliminar.to_csv(SALIDA_INTEGRADO, index=False, encoding="utf-8-sig")

# CSV auxiliar: integrado preliminar con trazabilidad.
integrado_trazabilidad.to_csv(SALIDA_TRAZABILIDAD, index=False, encoding="utf-8-sig")

# Reporte Excel con hojas/tablas solicitadas.
with pd.ExcelWriter(SALIDA_REPORTE_XLSX, engine="openpyxl") as writer:
    conteo_fuente.to_excel(writer, sheet_name="filas_por_fuente", index=False)
    conteo_area.to_excel(writer, sheet_name="filas_por_area", index=False)
    conteo_fuente_area.to_excel(writer, sheet_name="fuente_area", index=False)
    columnas_faltantes_df.to_excel(writer, sheet_name="columnas_faltantes", index=False)
    columnas_extra_df.to_excel(writer, sheet_name="columnas_extra", index=False)
    orden_canonico.to_excel(writer, sheet_name="orden_canonico", index=False)
    totales.to_excel(writer, sheet_name="totales", index=False)
    problemas_df.to_excel(writer, sheet_name="problemas", index=False)
    resumen_archivos.to_excel(writer, sheet_name="detalle_archivos", index=False)

# Ajuste visual simple del Excel, sin tocar datos.
try:
    from openpyxl import load_workbook

    wb = load_workbook(SALIDA_REPORTE_XLSX)
    for ws in wb.worksheets:
        ws.freeze_panes = "A2"
        for cell in ws[1]:
            cell.font = cell.font.copy(bold=True)
        for col_cells in ws.columns:
            col_letter = col_cells[0].column_letter
            max_len = max(len(str(cell.value)) if cell.value is not None else 0 for cell in col_cells)
            ws.column_dimensions[col_letter].width = min(max(max_len + 2, 12), 55)
    wb.save(SALIDA_REPORTE_XLSX)
except Exception as exc:
    print("Aviso: el reporte Excel se generó, pero no se pudo aplicar formato visual:", exc)

fecha = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
resumen_md = f"""
# Resumen de integración preliminar canónica

Fecha de ejecución: {fecha}

## Rutas

- Carpeta de entrada: `{ENTRADA_DIR}`
- Carpeta de salida: `{UNION_DIR}`
- Reporte Excel: `{SALIDA_REPORTE_XLSX}`

## Archivos de salida

- `integrado_preliminar_canonico.csv`: integrado con solo las 14 columnas canónicas.
- `integrado_preliminar_canonico_trazabilidad.csv`: integrado con columnas de trazabilidad.
- `resumen_integrado_preliminar.md`: este resumen.
- `reporte_integracion_preliminar.xlsx`: reporte estructural de integración.

## Alcance de esta libreta

Esta libreta solo une y valida estructura. No deduplica, no fusiona registros, no elimina registros salvo filas completamente vacías y no realiza limpieza profunda.

## Totales

{df_a_markdown(totales)}

## Conteo de filas por fuente

{df_a_markdown(conteo_fuente)}

## Conteo de filas por área

{df_a_markdown(conteo_area)}

## Columnas faltantes creadas

{df_a_markdown(columnas_faltantes_df)}

## Columnas extra separadas del integrado final

{df_a_markdown(columnas_extra_df)}

## Problemas detectados sin corregir

{df_a_markdown(problemas_df)}
""".strip()

SALIDA_RESUMEN_MD.write_text(resumen_md, encoding="utf-8")

print("Archivos generados:")
print("-", SALIDA_INTEGRADO)
print("-", SALIDA_TRAZABILIDAD)
print("-", SALIDA_RESUMEN_MD)
print("-", SALIDA_REPORTE_XLSX)

Archivos generados:
- C:\Users\hazar\Desktop\PROYECTO\02_modelo canonico\03_union\integrado_preliminar_canonico.csv
- C:\Users\hazar\Desktop\PROYECTO\02_modelo canonico\03_union\integrado_preliminar_canonico_trazabilidad.csv
- C:\Users\hazar\Desktop\PROYECTO\02_modelo canonico\03_union\resumen_integrado_preliminar.md
- C:\Users\hazar\Desktop\PROYECTO\outputs\reportes\reporte_integracion_preliminar.xlsx


C:\Users\hazar\AppData\Local\Temp\ipykernel_21388\1023821378.py:27: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  cell.font = cell.font.copy(bold=True)


## 8. Validación final rápida

El siguiente bloque confirma que el archivo principal quedó con exactamente 14 columnas y que la versión de trazabilidad conserva las 4 columnas auxiliares.

In [9]:
validacion_final = {
    "integrado_existe": SALIDA_INTEGRADO.exists(),
    "trazabilidad_existe": SALIDA_TRAZABILIDAD.exists(),
    "resumen_md_existe": SALIDA_RESUMEN_MD.exists(),
    "reporte_xlsx_existe": SALIDA_REPORTE_XLSX.exists(),
    "columnas_integrado_ok": list(integrado_preliminar.columns) == COLUMNAS_CANONICAS,
    "columnas_trazabilidad_ok": list(integrado_trazabilidad.columns) == COLUMNAS_CANONICAS + COLUMNAS_TRAZABILIDAD,
    "total_filas_integradas": len(integrado_preliminar),
}

validacion_final

{'integrado_existe': True,
 'trazabilidad_existe': True,
 'resumen_md_existe': True,
 'reporte_xlsx_existe': True,
 'columnas_integrado_ok': True,
 'columnas_trazabilidad_ok': True,
 'total_filas_integradas': 1471}

## Resultado esperado con los 8 archivos clasificados actuales

Con los reportes previos de clasificación de áreas, el total esperado del integrado preliminar es de **1,471 registros**, siempre que los 8 CSV clasificados tengan las mismas filas reportadas:

- ACM: 57
- EBSCO: 129
- Engineering Village: 142
- IEEE Xplore: 158
- ProQuest: 19
- ScienceDirect: 43
- Scopus: 561
- Web of Science: 362

La distribución final esperada por `Area` es:

- `ISBD`: 110
- `CC`: 141
- `IA`: 403
- `RS`: 189
- `TC`: 209
- `SIAV`: 419